In [1]:
from geopy.geocoders import Nominatim

In [2]:
geolocator = Nominatim(user_agent="kaggle_learn")
location = geolocator.geocode("Pyramid of Khufu")

print(location.point)
print(location.address)

29 58m 44.9927s N, 31 8m 3.16774s E
هرم خوفو, Cheops-Osirion-Sphinx Tunnel, كوم الأخضر, الجيزة, 12125, مصر


In [3]:
point = location.point

print("Latitude:", point.latitude)
print("Longitude:", point.longitude)

Latitude: 29.97916465
Longitude: 31.13421326082681


In [4]:
import numpy as np
import pandas as pd


TOP_UNIV_DATA = "../datasets/geospatial-learn-course-data/top_universities.csv"

universities = pd.read_csv(TOP_UNIV_DATA)
universities.head()

,Name
0,University of Oxford
1,University of Cambridge
2,Imperial College London
3,ETH Zurich
4,UCL


In [5]:
import geopandas as gpd
from tqdm.notebook import tqdm

tqdm.pandas()

def my_geocoder(row):
    try:
        point = geolocator.geocode(row).point
        return pd.Series({'Latitude': point.latitude, 'Longitude': point.longitude})
    except:
        return None
        
# universities[['Latitude', 'Longitude']] = universities.apply(lambda x: my_geocoder(x['Name']), axis=1)
universities[['Latitude', 'Longitude']] = universities.progress_apply(lambda x: my_geocoder(x['Name']), axis=1)
percent = (1 - (sum(np.isnan(universities["Latitude"])) / len(universities))) * 100
display(f"_{percent}_% of addresses were geocoded!")

# Drop universities that were not successfully geocoded
universities = universities.loc[~np.isnan(universities["Latitude"])]
universities = gpd.GeoDataFrame(universities, crs='epsg:4326',
                                geometry=gpd.points_from_xy(universities.Longitude, universities.Latitude))

universities.head()

  0%|          | 0/100 [00:00<?, ?it/s]

'_90.0_% of addresses were geocoded!'

,Name,Latitude,Longitude,geometry
0,University of Oxford,51.758708,-1.255668,POINT (-1.25567 51.75871)
1,University of Cambridge,52.210946,0.092005,POINT (0.092 52.21095)
2,Imperial College London,51.498959,-0.175641,POINT (-0.17564 51.49896)
3,ETH Zurich,47.413218,8.537491,POINT (8.53749 47.41322)
4,UCL,51.521785,-0.135151,POINT (-0.13515 51.52179)


In [11]:
from folium import Map, Marker

# Create a map
m = folium.Map(location=[54, 15], tiles='openstreetmap', zoom_start=2)
for idx, row in universities.iterrows():
    Marker([row['Latitude'], row['Longitude']], popup=row['Name']).add_to(m)
m

In [13]:
WORLD_DATA = '../datasets/geospatial-learn-course-data/naturalearth_lowres/naturalearth_lowres.shp'

world = gpd.read_file(WORLD_DATA)
europe = world.loc[world.continent == 'Europe'].reset_index(drop=True)

europe_stats = europe[["name", "pop_est", "gdp_md_est"]]
europe_boundaries = europe[["name", "geometry"]]
europe_boundaries.head()

,name,geometry
0,Russia,"MULTIPOLYGON (((178.7253 71.0988, 180 71.51571..."
1,Norway,"MULTIPOLYGON (((15.14282 79.67431, 15.52255 80..."
2,France,"MULTIPOLYGON (((-51.6578 4.15623, -52.24934 3...."
3,Sweden,"POLYGON ((11.02737 58.85615, 11.46827 59.43239..."
4,Belarus,"POLYGON ((28.17671 56.16913, 29.22951 55.91834..."


In [14]:
europe_stats.head()

,name,pop_est,gdp_md_est
0,Russia,142257519,3745000.0
1,Norway,5320045,364700.0
2,France,67106161,2699000.0
3,Sweden,9960487,498100.0
4,Belarus,9549747,165400.0


In [15]:
# Use an attribute join to merge data about countries in Europe
europe = europe_boundaries.merge(europe_stats, on="name")
europe.head()

,name,geometry,pop_est,gdp_md_est
0,Russia,"MULTIPOLYGON (((178.7253 71.0988, 180 71.51571...",142257519,3745000.0
1,Norway,"MULTIPOLYGON (((15.14282 79.67431, 15.52255 80...",5320045,364700.0
2,France,"MULTIPOLYGON (((-51.6578 4.15623, -52.24934 3....",67106161,2699000.0
3,Sweden,"POLYGON ((11.02737 58.85615, 11.46827 59.43239...",9960487,498100.0
4,Belarus,"POLYGON ((28.17671 56.16913, 29.22951 55.91834...",9549747,165400.0


In [18]:
# Use spatial join to match universities to countries in Europe
european_universities = gpd.sjoin(universities, europe)

print(f"We located {len(universities)} universities.")
print(f"Only {len(european_universities)} of the universities were located in Europe"
      f"(in {len(european_universities.name.unique())} different countries).")

european_universities.head()

We located 90 universities.
Only 86 of the universities were located in Europe(in 15 different countries).


,Name,Latitude,Longitude,geometry,index_right,name,pop_est,gdp_md_est
0,University of Oxford,51.758708,-1.255668,POINT (-1.25567 51.75871),28,United Kingdom,64769452,2788000.0
1,University of Cambridge,52.210946,0.092005,POINT (0.092 52.21095),28,United Kingdom,64769452,2788000.0
2,Imperial College London,51.498959,-0.175641,POINT (-0.17564 51.49896),28,United Kingdom,64769452,2788000.0
3,ETH Zurich,47.413218,8.537491,POINT (8.53749 47.41322),19,Switzerland,8236303,496300.0
4,UCL,51.521785,-0.135151,POINT (-0.13515 51.52179),28,United Kingdom,64769452,2788000.0
